In [1]:
import pandas as pd
import tpqoa as tpqoa

**Streaming Tick or Live Data**

In [2]:
api = tpqoa.tpqoa("D:\\Dissertation Mohit\\src\\oanda.cfg")

In [ ]:
api.stream_data("EUR_USD", stop=10)

**Orders and trades**

In [ ]:
api.create_order("EUR_USD", 100000)

In [ ]:
api.get_positions()

In [ ]:
api.create_order("EUR_USD", -100000)

In [ ]:
api.get_account_summary()

In [ ]:
api.get_transactions()

In [ ]:
api.print_transactions()

In [ ]:
order = api.create_order("EUR_USD", 100000, suppress=True, ret=True)

In [ ]:
order

In [ ]:
order['price']

In [ ]:
order['time']

**Trader class in live action**

In [17]:
from datetime import datetime, timedelta
import time
import numpy as np

In [18]:
import pickle 


In [22]:
model = pickle.load(open("D:\\Dissertation Mohit\\models\\model.pkl", "rb"))

Storing and resampling real-time tick data

In [ ]:
class GetTickData(tpqoa.tpqoa):
    
    def __init__(self, conf_file, instrument, bar_length, units, lags = 5):
        super().__init__(conf_file)
        self.instrument = instrument 
        self.bar_length = pd.to_timedelta(bar_length)
        self.tick_data = pd.DataFrame()
        self.data = pd.DataFrame() #New DataFrame to hold resampled bar data
        self.last_bar = pd.to_datetime(datetime.utcnow()).tz_localize('UTC') #Initialize last bar time to current time
        self.units = units
        self.position = 0
        self.profits = [] #New list to hold trade profits
        
        # **************************************Add strategy specific attributes here***************************************************
        self.window = 20 #Example attribute for a moving average window
        # ******************************************************************************************************************************
        
        # ***************************define your strategy here******************************
        self.data['returns'] = np.log(self.data['mid'] / self.data['mid'].shift(1))
        # self.data['boll'] = self.data['mid'].rolling(self.window).mean() + 2 * self.data['mid'].rolling(self.window).std()
        # self.data['min']  = self.data['mid'].rolling(self.window).min() / self.data['mid'] - 1
        # self.data['max'] = self.data['mid'].rolling(self.window).max() / self.data['mid'] - 1
        # self.data['mom'] = self.data['returns'].rolling(3).mean()
        # self.data['vol'] = self.data['returns'].rolling(self.window).std()
        self.data.dropna(inplace=True)
        
        cols = []
        for lag in range(1, self.lags + 1):
            col = 'lag_{}'.format(lag)
            self.data[col] = self.data['returns'].shift(lag)
            cols.append(col)
        self.data.dropna(inplace=True)
        
        self.data[cols] = (self.data[cols] - self.means) / self.std_devs
        
        self.data['position'] = model.predict(self.data[cols])
        # **********************************************************************************
        
    def on_success(self, time, bid, ask):
        print(self.ticks, end=" ")
        
        #collect and store tick data
        recent_tick = pd.to_datetime(time) #Pandas timestamp object
        df = pd.DataFrame({ "bid": [bid], "ask": [ask], "mid": [(bid + ask) / 2] }, index=[recent_tick])
    
        self.tick_data = pd.concat([self.tick_data, df])
        
        #If time longer than the bar length has passed since the last bar, resample and join the data
        if recent_tick - self.last_bar >= self.bar_length:
            self.resample_and_join()    
        
    def resample_and_join(self):
        self.data = self.tick_data.resample(self.bar_length, label='right').last().ffill().iloc[:-1]
        self.tick_data = self.tick_data.iloc[-1:] #Keep only the last tick for the next bar
        self.last_bar = self.data.index[-1] #Update the time of the last bar
        
    def execute_trades(self):
        if self.data["position"].iloc[-1] == 1:
            if self.position == 0:
                order = self.create_order(self.instrument, self.units, suppress = True, ret = True)
                self.report_trade(order, "GOING LONG")  # NEW
            elif self.position == -1:
                order = self.create_order(self.instrument, self.units * 2, suppress = True, ret = True) 
                self.report_trade(order, "GOING LONG")  # NEW
            self.position = 1
        elif self.data["position"].iloc[-1] == -1: 
            if self.position == 0:
                order = self.create_order(self.instrument, -self.units, suppress = True, ret = True)
                self.report_trade(order, "GOING SHORT")  # NEW
            elif self.position == 1:
                order = self.create_order(self.instrument, -self.units * 2, suppress = True, ret = True)
                self.report_trade(order, "GOING SHORT")  # NEW
            self.position = -1
        elif self.data["position"].iloc[-1] == 0: 
            if self.position == -1:
                order = self.create_order(self.instrument, self.units, suppress = True, ret = True) 
                self.report_trade(order, "GOING NEUTRAL")  # NEW
            elif self.position == 1:
                order = self.create_order(self.instrument, -self.units, suppress = True, ret = True)
                self.report_trade(order, "GOING NEUTRAL")  # NEW
            self.position = 0
            
    def report_trade(self, order, going):  # NEW
        time = order["time"]
        units = order["units"]
        price = order["price"]
        pl = float(order["pl"])
        self.profits.append(pl)
        cumpl = sum(self.profits)
        print("\n" + 100* "-")
        print("{} | {}".format(time, going))
        print("{} | units = {} | price = {} | P&L = {} | Cum P&L = {}".format(time, units, price, pl, cumpl))
        print(100 * "-" + "\n") 

In [5]:
td = GetTickData("D:\\Dissertation Mohit\\src\\oanda.cfg","EUR_USD", "5s")

In [6]:
td.stream_data("EUR_USD", stop=20)

1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 

In [8]:
td.tick_data

,bid,ask,mid
2026-03-19 15:02:00.340905032+00:00,1.15138,1.15145,1.151415
2026-03-19 15:02:00.402013177+00:00,1.15136,1.15143,1.151395
2026-03-19 15:02:00.833587422+00:00,1.15131,1.15138,1.151345
2026-03-19 15:02:01.752246542+00:00,1.15135,1.15144,1.151395
2026-03-19 15:02:01.953087148+00:00,1.15138,1.15145,1.151415
2026-03-19 15:02:02.293594927+00:00,1.15139,1.15147,1.151430
2026-03-19 15:02:02.854771071+00:00,1.15139,1.15145,1.151420
2026-03-19 15:02:02.902474355+00:00,1.15139,1.15147,1.151430
2026-03-19 15:02:03.411300073+00:00,1.15140,1.15146,1.151430
2026-03-19 15:02:03.626206576+00:00,1.15144,1.15151,1.151475


In [7]:
td.data

,bid,ask,mid
2026-03-19 15:02:00+00:00,1.15141,1.15149,1.15145


**Working with historical data and real time tick data**

In [11]:
now = datetime.utcnow()
now

datetime.datetime(2026, 3, 19, 15, 4, 17, 171466)

In [13]:
now = now -timedelta(microseconds=now.microsecond)
now

datetime.datetime(2026, 3, 19, 15, 4, 17)

In [15]:
yesterday = now - timedelta(days=1)
yesterday

datetime.datetime(2026, 3, 18, 15, 4, 17)